# DAT494, Lab 3: GPT From Scratch

Your goal in this lab is to independently reproduce some of our work in class on implementing the GPT-2 model.

You are strongly encouraged to complete this lab with minimum external assistance: Doing so will help you internalize the fundamentals of a decoder-based transformer model.

The following cells outline the basic steps needed to implement GPT-2, and follow the posted notebooks, with some modifications in the overall sequence. You may also wish consult the following as additional references:

-  Build a Large Language Model (From Scratch), 1st ed, by Sebastian Raschka. Manning. 2025. Online via https://libguides.asu.edu/Oreilly
- GitHub repo for GPT-2: https://github.com/openai/gpt-2


In [38]:
#Basic Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Basic PyTorch Libraries
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss

from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Part 1: Tokenizing and Chunking Text

- Get an appropriate GPT-2 byte-pair tokenizer (I suggest the HuggingFace version)
- Confirm that you can encode and decode sample text
- Get some kind of text training set of your choice, e.g., large books from Project Gutenberg, a Wikipedia dump, etc.
- Confirm that you can chunk this data into input/output sequences for next token ahead prediction
- Wrap into PyTorch Datasets and DataLoaders

In [39]:
#import transformers from Hugging face to get GPT2 tokenizer

import transformers
from transformers import GPT2TokenizerFast

print(f"HuggingFace Transformers version: {transformers.__version__}")

Tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

HuggingFace Transformers version: 5.0.0


In [40]:
MyInputText = "This class covers two or three graduate course portions"

#encoding to get token IDs
TokenID = Tokenizer.encode(MyInputText)
print(TokenID)

#decoding to confirm encoded text is the input text
DecodedText = Tokenizer.decode(TokenID)
print(DecodedText)



[1212, 1398, 8698, 734, 393, 1115, 10428, 1781, 16690]
This class covers two or three graduate course portions


In [41]:
#Importing a book as a file  from Project Gutenberg for training data using gutenberg URL
#obtained URL by google search and colab autofill
import urllib.request

GutenbergURL = "https://www.gutenberg.org/files/1342/1342-0.txt"
urllib.request.urlretrieve(GutenbergURL, "PrideAndPrejudice.txt")

with open("PrideAndPrejudice.txt", "r", encoding="utf-8") as f:
    PrideAndPrejudice = f.read()
print(f"Downloaded {len(PrideAndPrejudice):,} characters from {GutenbergURL}")

Downloaded 728,846 characters from https://www.gutenberg.org/files/1342/1342-0.txt


In [42]:
EntirebookTokens = Tokenizer.encode(PrideAndPrejudice)
#Creating a tensor to feed tokens into models for training
EntirebookTokens = torch.tensor(EntirebookTokens)
print(f"EntirebookTokens shape: {EntirebookTokens.shape}")

NumberofTokens = len(EntirebookTokens)
print(f"The book has {NumberofTokens:,} tokens")

#Creating Input and output seq using chunking.
#Match the model's reduced context length to avoid indexing errors
ChunkSize = 512
NumberofChunks = NumberofTokens // ChunkSize
print(f"The book has {NumberofChunks:,} chunks")

input_seq = EntirebookTokens[:NumberofChunks*ChunkSize].view(-1, ChunkSize)
ouput_seq = EntirebookTokens[1:NumberofChunks*ChunkSize+1].view(-1, ChunkSize)

print(f"input_seq shape: {input_seq.shape}")
print(f"ouput_seq shape: {ouput_seq.shape}")

Token indices sequence length is longer than the specified maximum sequence length for this model (191673 > 1024). Running this sequence through the model will result in indexing errors


EntirebookTokens shape: torch.Size([191673])
The book has 191,673 tokens
The book has 374 chunks
input_seq shape: torch.Size([374, 512])
ouput_seq shape: torch.Size([374, 512])


In [43]:
#creating a token dataset using the input_seq and output_seq created from the book tokens and split that into training and validation data sets
#using 80-20 training, validation split
TokenDataset = torch.utils.data.TensorDataset(input_seq, ouput_seq)
n_train = int(0.8  * len(TokenDataset))
n_val = len(TokenDataset) - n_train
train_set, val_set = torch.utils.data.random_split(TokenDataset, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False)

#input tokens and the expected prediction
x_batch, y_batch = next(iter(train_loader))
print(f"x_batch shape: {x_batch.shape}")
print(f"y_batch shape: {y_batch.shape}")

x_batch shape: torch.Size([8, 512])
y_batch shape: torch.Size([8, 512])


## Part 2: Embedding Text

- Confirm that you can perform the following embeddings into a vector space of dimension `d_model`

1. Token embedding
2. Positional embedding, allow up to `context_length` positions

- Then, confirm you can sum the two for an overall sequence embedding

In [44]:
#parameters for vocab size, model dimensions and context length
Vocab_size = Tokenizer.vocab_size
Context_length = 1024
d_model = 768

print(f"Vocab_size: {Vocab_size}")

#Token embedding & Positional embedding
Token_embedding = nn.Embedding(Vocab_size, d_model)
sample_token_embedding = Token_embedding(torch.tensor([1212, 1398, 8698, 734, 393, 1115, 10428, 1781, 16690]))
print(f"Token embedding shape: {sample_token_embedding.shape}")

#Create positional table for context length
Position_table = nn.Embedding(Context_length, d_model)
sample_position_embedding = Position_table(torch.arange(len(TokenID))) # Fix: Pass the length of TokenID to arange
print(f"Positional embedding shape: {sample_position_embedding.shape}")


#sequence embedding
sample_sequence_embedding = sample_token_embedding + sample_position_embedding
print(f"Sequence embedding shape: {sample_sequence_embedding.shape}")


Vocab_size: 50257
Token embedding shape: torch.Size([9, 768])
Positional embedding shape: torch.Size([9, 768])
Sequence embedding shape: torch.Size([9, 768])


## Part 3:Dummy GPT Model

- Create a dummy GPT Model class, as well as the following necessary dummy classes:

1. Dummy Multi-Headed Causal Attention
2. Dummy Layer Normalization
3. Dummy Feed-Forward Block
4. Dummy Overall Transformer Block

- Do not forget to include Dropout layers for a true GPT-2 clone

- Use the above dummy classes, along with your text preparation and embedding steps, to confirm that the dummy model can process arbitrary input sequences (of length up to the context window)


In [45]:
#nn.LayerNorm already performs layer normalization. creating a class using model param
class DummyLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x)

In [46]:
#Creating feedforward block class -- 2 layer MLP like GPT-2
# layer 1 expands to 4*d_model; layer 2 contracts to d_model
class DummyFeedForward(nn.Module):
  def __init__(self, d_model):
    super().__init__()
    self.linear1 = nn.Linear(d_model, 4*d_model)
    self.linear2 = nn.Linear(4*d_model, d_model)

  def forward(self, x):
    x = self.linear1(x)
    x = nn.GELU()(x)
    x = self.linear2(x)
    x = nn.Dropout(0.1)(x)
    return x



In [47]:
import torch
import torch.nn as nn
import numpy as np

"""
Attention calculation uses QKV, in particular K transpose.  Attention(Q,K,V) = softmax(Q*K^T/sqrt(d_k))*V
So compute Q, K, V using linear layers. Then plug in Q,K,V into the above expression for Attention calculation
1/sqrt(d_k) is the scaling factor. Keeping in mind, in the decoder, masked attention is what we calculate, we
apply the appropriate masking to the attention matrix. Number of heads is a parameter that allows experimenting
with different number of heads in the multi-head attention mechanism.

"""
class DummyMultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads, context_length): # Added context_length
    super().__init__()
    self.d_model = d_model
    self.n_heads = n_heads
    self.context_length = context_length # Stored context_length
    self.Q = nn.Linear(
        d_model, d_model, bias=False)
    self.K = nn.Linear(
        d_model, d_model, bias=False)
    self.V = nn.Linear(
        d_model, d_model, bias=False)
    self.O = nn.Linear(
        d_model, d_model, bias=False)
    self.softmax = nn.Softmax(dim=-1)
    d_k = d_model // n_heads
    self.scale = 1/np.sqrt(d_k)
    # Register the mask as a buffer so it automatically moves with the model to the correct device
    # Use self.context_length here
    self.register_buffer('mask', torch.tril(torch.ones(self.context_length, self.context_length)).view(1, 1, self.context_length, self.context_length))

  def forward(self, x):

    # Obtain Q,K,V through the linear layers
    Q = self.Q(x)
    K = self.K(x)
    V = self.V(x)

    # realign the tensors for matrix multiplication to calculate attention scores and to slice the linear layer
    # output per head
    Q = Q.view(Q.shape  [0], Q.shape[1], self.n_heads, self.d_model//self.n_heads)
    K = K.view(K.shape  [0], K.shape[1], self.n_heads, self.d_model//self.n_heads)
    V = V.view(V.shape  [0], V.shape[1], self.n_heads, self.d_model//self.n_heads)
    K = K.transpose(1,2)
    Q = Q.transpose(1,2)
    V = V.transpose(1,2)

    # Caculate attention score based on the formula above
    attention = torch.matmul(Q, K.transpose(2,3)) * self.scale
    attention = attention.masked_fill(self.mask[:,:,:attention.shape[-2],:attention.shape[-1]] == 0, float('-inf'))
    attention = self.softmax(attention)
    out = torch.matmul(attention, V)

    #merge the per head attention output into 1 model_dim sized vector for the final linear output projection
    out = out.transpose(1,2).contiguous()
    out = out.view(out.shape[0], out.shape[1], self.d_model)
    out = self.O(out)
    return out

In [48]:
# Dummy Transformer block to put together the components defined above
# the transformer is a sequence of layernorm followed by Multi-head attention
# and then a layernorm followed by feedforward. Making residual connections as needed

class DummyTransformerBlock(nn.Module):
  def __init__(self, d_model, n_heads, context_length):
    super().__init__()
    self.d_model = d_model
    self.n_heads = n_heads

    # Pass context_length to the attention layer
    self.attention = DummyMultiHeadAttention(d_model, n_heads, context_length)
    self.norm1 = DummyLayerNorm(d_model)
    self.feedforward = DummyFeedForward(d_model)
    self.norm2 = DummyLayerNorm(d_model)
    self.dropout = nn.Dropout(0.1)

  def forward(self, x):
    x = self.norm1(x)
    x = x + self.dropout(self.attention(x))
    x = self.norm2(x)
    x = x + self.dropout(self.feedforward(x))
    return x

In [49]:
# Putting together a Dummy GPT Model using the transformer.
class DummyGPT(nn.Module):
  def __init__(self, vocab_size, context_length, d_model, n_heads, n_blocks):
    super().__init__()
    self.d_model = d_model
    self.n_heads = n_heads
    self.n_blocks = n_blocks
    self.vocab_size = vocab_size
    self.context_length = context_length

    self.token_embedding = nn.Embedding(vocab_size, d_model)
    self.position_embedding = nn.Embedding(context_length, d_model)
    self.emb_dropout = nn.Dropout(0.1)

    # Pass context_length to each transformer block
    self.blocks = nn.Sequential(*[DummyTransformerBlock(d_model, n_heads, context_length) for _ in range(n_blocks)])
    self.layer_norm = DummyLayerNorm(d_model)
    self.output_head = nn.Linear(d_model, vocab_size)

  def forward(self, x):
    Batch, Chunk_length = x.shape
    if Chunk_length > self.context_length:
        raise ValueError(f"Input sequence length {Chunk_length} exceeds context length {self.context_length}")

    x = self.token_embedding(x)
    position_ids = torch.arange(Chunk_length, device=x.device)
    position_ids = position_ids.expand(Batch, Chunk_length)
    position_embedding = self.position_embedding(position_ids)
    x = x + position_embedding
    x = self.emb_dropout(x)

    x = self.blocks(x)
    x = self.layer_norm(x)
    logits = self.output_head(x)
    return logits

## Part 4: Fill in All Necessary Blocks

- Fill in all your dummy layers/blocks above, and deliver a final GPT model that is ready for training


In [50]:
# confirming the dummy model can process sequences of arbitrary length upto size context length
# Using a short, medium and long sequence(size of context window) to see if the model is able process these sequences

vocab_size = Tokenizer.vocab_size
context_length = 512
d_model = 128
n_heads = 4
n_blocks = 12
short_seq = torch.randint(0, vocab_size, (1, 10))
medium_seq = torch.randint(0, vocab_size, (1, 256))
long_seq = torch.randint(0, vocab_size, (1, context_length))
dummy_model = DummyGPT(vocab_size, context_length, d_model, n_heads, n_blocks)
short_logits = dummy_model(short_seq)
print(f"short seq shape: {short_seq.shape}")
print(f"short logits shape: {short_logits.shape}")
medium_logits = dummy_model(medium_seq)
print(f"medium seq shape: {medium_seq.shape}")
print(f"medium logits shape: {medium_logits.shape}")
long_logits = dummy_model(long_seq)
print(f"long seq shape: {long_seq.shape}")
print(f"long logits shape: {long_logits.shape}")

# using the real sample input text created above to check the model
# Convert TokenID list to a tensor before passing it to the model
TokenID_tensor = torch.tensor(TokenID, dtype=torch.long).unsqueeze(0)
sample_logits = dummy_model(TokenID_tensor)
print(f"sample seq shape: {TokenID_tensor.shape}")
print(f"sample logits shape: {sample_logits.shape}")

short seq shape: torch.Size([1, 10])
short logits shape: torch.Size([1, 10, 50257])
medium seq shape: torch.Size([1, 256])
medium logits shape: torch.Size([1, 256, 50257])
long seq shape: torch.Size([1, 512])
long logits shape: torch.Size([1, 512, 50257])
sample seq shape: torch.Size([1, 9])
sample logits shape: torch.Size([1, 9, 50257])


In [51]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, loss_val):
        if self.best_loss is None:
            self.best_loss = loss_val
        elif loss_val > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = loss_val
            self.counter = 0

## Part 5: Next Token Ahead Pre-Training

(Pre-) train the model on your chosen corpus using next token ahead prediction. Use reasonable structural parameters for the model that allow you to train over a sensible amount of time. You may want to use a GPU (e.g., Google Colab) for training, but this is not required.

In [52]:
model = DummyGPT(vocab_size, context_length, d_model, n_heads, n_blocks).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)  # Better than adam due to how weight decay is directly applied to parameters
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)  # Halves learning rate if there is no improvement over 2 epochs
stopper = EarlyStopping(patience=5)

num_epochs = 50
loss_func = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    # Training Loss Calculation
    model.train()
    train_losses = []
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = loss_func(logits.view(-1, model.vocab_size), y.view(-1))
        # Logits need to be reshaped so that the output tokens can be aligned
        # to target labels

        optimizer.zero_grad()
        loss.backward()

        # Implementing gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        train_losses.append(loss.item())

    # Validation Loss Calculation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for X, Y in val_loader:
            X, Y = X.to(device), Y.to(device)
            logits = model(X)
            v_loss = loss_func(logits.view(-1, model.vocab_size), Y.view(-1))
            val_losses.append(v_loss.item())

    avg_train_loss = sum(train_losses) / len(train_losses)
    avg_val_loss = sum(val_losses) / len(val_losses)

    print(f"Epoch: {epoch}, Average Training Loss: {avg_train_loss:.4f}, Average Validation Loss: {avg_val_loss:.4f}")

    scheduler.step(avg_val_loss)
    stopper(avg_val_loss)

    if stopper.early_stop:
        print(f"Early stopping triggered at epoch {epoch}")
        break

Epoch: 0, Average Training Loss: 7.9962, Average Validation Loss: 6.3417
Epoch: 1, Average Training Loss: 5.9307, Average Validation Loss: 5.8985
Epoch: 2, Average Training Loss: 5.5202, Average Validation Loss: 5.5514
Epoch: 3, Average Training Loss: 5.1692, Average Validation Loss: 5.3232
Epoch: 4, Average Training Loss: 4.9563, Average Validation Loss: 5.1738
Epoch: 5, Average Training Loss: 4.7487, Average Validation Loss: 5.0674
Epoch: 6, Average Training Loss: 4.6102, Average Validation Loss: 5.0051
Epoch: 7, Average Training Loss: 4.4726, Average Validation Loss: 4.9465
Epoch: 8, Average Training Loss: 4.3456, Average Validation Loss: 4.9002
Epoch: 9, Average Training Loss: 4.1962, Average Validation Loss: 4.8922
Epoch: 10, Average Training Loss: 4.1024, Average Validation Loss: 4.8522
Epoch: 11, Average Training Loss: 3.9837, Average Validation Loss: 4.8656
Epoch: 12, Average Training Loss: 3.8682, Average Validation Loss: 4.8641
Epoch: 13, Average Training Loss: 3.7513, Averag

## Part 6: Autoregressive Text Generation

- Implement a function or method to autoregressively generate text with your model, starting from some seed text
- Allow greedy or stochastic decoding, and specify temperature and top-k (possible None) parameters for decoding
- Play around with generating text, include some examples you find interesting

In [57]:
def text_gen(model, tokenizer, seed_text, max_new_tokens=50, context_length=512, temperature=1.0, top_k=None, greedy=False):
    model.eval()
    input_tokens = tokenizer.encode(seed_text)
    input_tensor = torch.tensor(input_tokens, dtype=torch.long).unsqueeze(0).to(device)

    for i in range(max_new_tokens):
        condensed_tensor = input_tensor[:, -context_length:]

        with torch.no_grad():
            logits = model(condensed_tensor)
            logits = logits[:, -1, :] / temperature

            if greedy:
                # Greedy decoding picks the token with the maximum value (hence the use of argmax)
                next_token = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                # Applies softmax function so that future tokens can be sampled from probability distribution
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float('Inf')

                # Sampling and creation of the probability distribution
                probs = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

            input_tensor = torch.cat((input_tensor, next_token), dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_tensor.squeeze().tolist())

print("Greedy Decoding:\n")
print(text_gen(model, Tokenizer, "The surest way to disappoint him will be", greedy=True))
print("\n")

print("Stochastic Decoding:\n")
print(text_gen(model, Tokenizer, "The surest way to disappoint him will be", top_k=50, temperature=0.8))

Greedy Decoding:

The surest way to disappoint him will be
” said she was not to her.


“I am sure,” said she was not to her.” said Elizabeth, “I am
that_, and I am not
that_, and


Stochastic Decoding:

The surest way to disappoint him will be
could to make that Mr. Darcy’s marriage. Darcy had
to Elizabeth, and she could not much as he
at himself. She then had been done so on his sister; and as she was not
to her


## Optional Part 7: Fine-Tune for Classification

Optionally, you may replace the final language modeling head of your model with a classification head, and fine-tune the model for text classification. A simple example is sentiment analysis on the IMDB Database, but many other classification datasets are available as well.

## Optional Part 8: Supervised Fine-Tuning on Instruction Data

Also optionally, you may follow our work in class to perform supervised fine-tuning on an instruction dataset, such as the Alpaca Instruction Set (https://crfm.stanford.edu/2023/03/13/alpaca.html).